# LFM Instance Segmentation Example Workflow
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data. 

## Purpose of this notebook
This notebook is designed to be used as an example semantic segmentation workflow ("crater" vs "non crater" model prediction). A pretrained DinoV3 model is loaded from disk, and we build our own crater detection model on top of this. The model loads data from the lfm project space, then runs several epochs of training on this data, and finally visualizes the model performance on the validation dataset. If you would like to control some of the model parameters, see the "User Configuration" section below.  

**Note**: currently, the training dataset used in this notebooks is comprised of 12-band input data (5 VIS bands, 2 UV bands, 5 Kaguya static bands). If you wish to train your own model with a different input dataset (i.e. different static band configuration), that will be added soon. If you would like to filter out certain WAC/Static bands, you can filter them using the BAND_FILTER variable in the config. **See the README in the [LFM repo](https://github.com/nasa-nccs-hpda/lfm) for more info.** 

## Imports, Dino Repo Clone

In [ ]:
import os
import sys
import torch
import subprocess
import warnings
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

%matplotlib inline

In [ ]:
# Get the repo root directory (parent of notebooks/)
repo_root = Path.cwd().parent

# Convert /panfs path to /explore path (JupyterHub quirk)
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)

# Verify we're in the right location
if not (repo_root / "lfm").exists():
    raise FileNotFoundError(
        "Cannot find lfm/ directory. "
        "Please ensure you're running this notebook from the lfm/notebooks/ directory."
    )

# Add the parent of the repo to sys.path for imports
sys.path.insert(0, str(repo_root.parent))

# Import required modules
from lfm.lfm.tasks.inst_segmentation.iseg_dataset import get_dataloaders, calculate_dataset_statistics
from lfm.lfm.tasks.inst_segmentation.iseg_driver import train_model
from lfm.lfm.tasks.inst_segmentation.iseg_model import create_mask2former_dinov3_model, load_dinov3_encoder
from lfm.lfm.tasks.all_tasks.all_utils import prepare_output_dir
print("✓ Successfully imported LFM modules")

## User Configuration

#### Paths
`INPUT_ROOT_DIR`: there are multiple different dataset configurations under the lfm/model_inputs/300_300_inputs folder. Choose any of the band subdirectories in the 300_300_inputs folder, and leave the other variables under "data paths" (IMAGE_DIR, LABEL_DIR, etc) as they are. 

`OUTPUT_DIR`: this is a relative path, so by default the outputs (visualizations, model checkpoints, dataset statistics) will go to the same folder as the notebook. 

#### Dataset parameters
`MAX_SAMPLES`: number of training samples to look for in INPUT_ROOT_DIR. 

`TRAIN_SPLIT`: weight of how many training samples versus validation samples; the default is 0.8, or 80% training.

#### Training hyperparameters
`BATCH_SIZE`: best to leave this at 16 to conserve VRAM, especially at higher number of input bands (>7). 

`NUM_EPOCHS`: default is 100, the model tends to start to get its best results around here. Feel free to experiment with this.

`BASE_LR`,`WEIGHT_DECAY`: feel free to change these to adjust how aggressively the model tries to tune to each training batch. Higher means more aggressive model learning, but can also mean the model has a harder time converging to the correct result. 

#### Model hyperparameters
`FREEZE_ENCODER`: whether to keep the Dino backbone frozen; we found better results with False, but feel free to try with freezing set to true. 

`NUM_BANDS`: number of bands to include in the input. Currently supported are 3/5/7/12-band inputs.

`BAND_FILTER`: list of band indices (0-indexed) to use in the dataset; for example, if I want to use only VIS bands, I would supply [0, 1, 2, 3, 4]. Band ordering is the following for the 12-band dataset: VIS (0-4), UV (5, 6), KAGUYA STATIC (7-11). 

In [ ]:
# Data path
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/inst_seg"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16  # Number of images fed into the model at a time
NUM_EPOCHS = 100  # 10 is the default, increase for more model learning
BASE_LR = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer

# Model parameters
FREEZE_ENCODER = False  # Whether to keep the DinoV3 weights frozen
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Bands to keep, in order of filtering

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Create output directory
This will contain model checkpoints and visualizations. It will ask you to overwrite the directory if it already exists, so ensure you copy files that you want to keep to a different directory!

In [ ]:
prepare_output_dir(OUTPUT_DIR)

## Create dataloaders

In [ ]:
print("\nSTEP 3: Creating dataloaders.")
print("="*60)

train_loader, val_loader, weight_assignments = get_dataloaders(
    base_dir=INPUT_ROOT_DIR,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR,
    debug=True,
    band_filter=BAND_FILTER,
    output_dir=OUTPUT_DIR,
)

print(f"    Train batches: {len(train_loader)}")
print(f"    Val batches: {len(val_loader)}")

## Load Encoder and Create Model

In [ ]:
print("\n" + "="*60)
print("Loading DINO encoder and creating model.")
print("="*60)

encoder = load_dinov3_encoder(device=device)

In [ ]:
print("\n" + "="*60)
print("Loading mask2former Dino model...")
print("="*60)

model = create_mask2former_dinov3_model(
    encoder=encoder,
    freeze_backbone=FREEZE_ENCODER,
    device=device,
    weight_assignments=weight_assignments,
)

## Run Training

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    device=device,
)

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.show()